# 01 — Ingest Online Retail Dataset into Bronze

Project: Databricks SQL Retail Analytics Dashboard  
Layer: Bronze  
Source: UCI Online Retail Dataset  
Target Table: retail_analytics_project.bronze_online_retail_raw

This notebook downloads the Online Retail dataset from the UCI Machine Learning Repository and stores the raw transactional records into a Bronze Delta table.

The Bronze layer keeps the source data as close as possible to its original structure while adding ingestion metadata for traceability.

In [0]:
%pip install openpyxl

In [0]:
%restart_python

In [0]:
import os
import zipfile
import requests
import pandas as pd

from pyspark.sql.functions import (
    current_timestamp,
    lit,
    col,
    trim
)

In [0]:
schema_name = "retail_analytics_project"
bronze_table = f"{schema_name}.bronze_online_retail_raw"

dataset_url = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"

local_zip_path = "/tmp/online_retail.zip"
extract_dir = "/tmp/online_retail"
excel_file_path = f"{extract_dir}/Online Retail.xlsx"

print(f"Schema: {schema_name}")
print(f"Bronze table: {bronze_table}")
print(f"Dataset URL: {dataset_url}")

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")
spark.sql(f"USE SCHEMA {schema_name}")

print(f"Schema ready: {schema_name}")

In [0]:
response = requests.get(dataset_url, timeout=60)
response.raise_for_status()

with open(local_zip_path, "wb") as file:
    file.write(response.content)

print(f"Dataset downloaded successfully: {local_zip_path}")
print(f"File size: {round(os.path.getsize(local_zip_path) / (1024 * 1024), 2)} MB")

In [0]:
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(local_zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print("Files extracted:")
for file_name in os.listdir(extract_dir):
    print(f"- {file_name}")

In [0]:
import pandas as pd

retail_pdf = pd.read_excel(
    excel_file_path,
    engine="openpyxl"
)

# Keep only expected columns
retail_pdf = retail_pdf[
    [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country"
    ]
]

# Helper functions
def clean_string(value):
    return None if pd.isna(value) else str(value).strip()

def clean_int(value):
    return None if pd.isna(value) else int(value)

def clean_float(value):
    return None if pd.isna(value) else float(value)

def clean_datetime_as_string(value):
    if pd.isna(value):
        return None
    
    parsed_value = pd.to_datetime(value, errors="coerce")
    
    if pd.isna(parsed_value):
        return None
    
    return parsed_value.strftime("%Y-%m-%d %H:%M:%S")


# Text / identifier columns
retail_pdf["InvoiceNo"] = retail_pdf["InvoiceNo"].apply(clean_string)
retail_pdf["StockCode"] = retail_pdf["StockCode"].apply(clean_string)
retail_pdf["Description"] = retail_pdf["Description"].apply(clean_string)
retail_pdf["Country"] = retail_pdf["Country"].apply(clean_string)

# Numeric columns
retail_pdf["Quantity"] = pd.to_numeric(retail_pdf["Quantity"], errors="coerce")
retail_pdf["Quantity"] = retail_pdf["Quantity"].apply(clean_int)

retail_pdf["UnitPrice"] = pd.to_numeric(retail_pdf["UnitPrice"], errors="coerce")
retail_pdf["UnitPrice"] = retail_pdf["UnitPrice"].apply(clean_float)

# CustomerID is an identifier, not a measure
retail_pdf["CustomerID"] = pd.to_numeric(retail_pdf["CustomerID"], errors="coerce")
retail_pdf["CustomerID"] = retail_pdf["CustomerID"].apply(
    lambda x: None if pd.isna(x) else str(int(x))
)

# Convert InvoiceDate to string first.
# Later we will convert it to timestamp in Spark.
retail_pdf["InvoiceDate"] = retail_pdf["InvoiceDate"].apply(clean_datetime_as_string)

print(f"Rows: {retail_pdf.shape[0]}")
print(f"Columns: {retail_pdf.shape[1]}")

display(retail_pdf.head(10))

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

from pyspark.sql.functions import to_timestamp, col

retail_schema = StructType([
    StructField("InvoiceNo", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", StringType(), True),
    StructField("UnitPrice", DoubleType(), True),
    StructField("CustomerID", StringType(), True),
    StructField("Country", StringType(), True)
])

bronze_records = retail_pdf.to_dict("records")

bronze_df_raw = spark.createDataFrame(
    bronze_records,
    schema=retail_schema
)

bronze_df = bronze_df_raw.withColumn(
    "InvoiceDate",
    to_timestamp(col("InvoiceDate"), "yyyy-MM-dd HH:mm:ss")
)

display(bronze_df.limit(10))

In [0]:
from pyspark.sql.functions import current_timestamp, lit

bronze_df = (
    bronze_df
    .withColumn("source_system", lit("uci_online_retail"))
    .withColumn("source_file", lit("Online Retail.xlsx"))
    .withColumn("source_url", lit(dataset_url))
    .withColumn("bronze_ingestion_ts", current_timestamp())
)

display(bronze_df.limit(10))

In [0]:
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(bronze_table)

print(f"Bronze table created successfully: {bronze_table}")

In [0]:
bronze_count = spark.table(bronze_table).count()

print(f"Bronze record count: {bronze_count}")

display(
    spark.sql(f"""
    SELECT *
    FROM {bronze_table}
    LIMIT 10
    """)
)

In [0]:
spark.table(bronze_table).printSchema()

In [0]:
display(
    spark.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT InvoiceNo) AS distinct_invoices,
        COUNT(DISTINCT StockCode) AS distinct_products,
        COUNT(DISTINCT CustomerID) AS distinct_customers,
        MIN(InvoiceDate) AS min_invoice_date,
        MAX(InvoiceDate) AS max_invoice_date,
        SUM(CASE WHEN CustomerID IS NULL THEN 1 ELSE 0 END) AS rows_without_customer_id,
        SUM(CASE WHEN Description IS NULL THEN 1 ELSE 0 END) AS rows_without_description,
        SUM(CASE WHEN Quantity <= 0 THEN 1 ELSE 0 END) AS rows_with_non_positive_quantity,
        SUM(CASE WHEN UnitPrice <= 0 THEN 1 ELSE 0 END) AS rows_with_non_positive_unit_price
    FROM {bronze_table}
    """)
)

In [0]:
display(
    spark.sql(f"""
    SELECT
        CASE
            WHEN CAST(InvoiceNo AS STRING) LIKE 'C%' THEN 'Cancellation'
            ELSE 'Regular Transaction'
        END AS transaction_type,
        COUNT(*) AS total_rows
    FROM {bronze_table}
    GROUP BY
        CASE
            WHEN CAST(InvoiceNo AS STRING) LIKE 'C%' THEN 'Cancellation'
            ELSE 'Regular Transaction'
        END
    ORDER BY total_rows DESC
    """)
)

In [0]:
display(
    spark.sql(f"""
    SELECT
        InvoiceNo,
        StockCode,
        Description,
        Quantity,
        InvoiceDate,
        UnitPrice,
        CustomerID,
        Country
    FROM {bronze_table}
    WHERE CAST(InvoiceNo AS STRING) LIKE 'C%'
    LIMIT 20
    """)
)

In [0]:
display(
    spark.sql(f"""
    SELECT
        Country,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT InvoiceNo) AS total_invoices,
        COUNT(DISTINCT CustomerID) AS total_customers
    FROM {bronze_table}
    GROUP BY Country
    ORDER BY total_rows DESC
    LIMIT 20
    """)
)